# NearMiss: train an overhead (drone) detector on VisDrone

This trains the detector NearMiss uses for drone footage. It runs on Google's free GPU, not your computer.

**VisDrone** is a public dataset of about 6,500 drone images with hand-drawn boxes around cars, vans, trucks, buses,
pedestrians, bicycles and motorcycles (Zhu et al., *Detection and Tracking Meet Drones Challenge*, IEEE TPAMI 2021).
Unlike the stock aerial model, a model trained on it also finds **pedestrians and cyclists from above**.

### How to run
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** Allow Google Drive access when asked: checkpoints are saved there, so a disconnect does not lose the training.
3. Training takes about **2–3 hours**. Keep this tab open. If Colab disconnects, reconnect and run the cells marked *Resume*.
4. The last cell downloads **`nearmiss-visdrone.pt`**. Put it in the NearMiss folder (next to `app.py`).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."

## 2. Install Ultralytics
The same version the NearMiss app uses, so the trained weights load there.

In [ ]:
%pip install -q ultralytics==8.4.183

## 3. Save checkpoints to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/nearmiss_training"
import os
os.makedirs(PROJECT, exist_ok=True)
print("Checkpoints go to", PROJECT)

## 4. Train

* Starts from `yolo11s.pt` (pretrained on everyday photos), so it only has to learn the drone view.
* `imgsz=960` matches the size NearMiss detects at.
* The first run downloads VisDrone (about 2 GB) and converts it. That takes about 5 minutes.
* A checkpoint is saved to Drive every 5 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")
model.train(
    data="VisDrone.yaml",
    imgsz=960,
    epochs=30,
    batch=8,
    patience=10,          # stop early if it stops improving
    cos_lr=True,
    close_mosaic=5,
    save_period=5,
    project=PROJECT,
    name="visdrone",
    exist_ok=True,
    workers=2,
)

### Resume (only if Colab disconnected)
Run cells 1–3 again, then this cell instead of cell 4. It continues from the last checkpoint on Drive.

In [ ]:
from ultralytics import YOLO
YOLO(f"{PROJECT}/visdrone/weights/last.pt").train(resume=True)

## 5. How good is it?
mAP50 is the share of road users found with a box that overlaps the hand-drawn one by at least half (1.0 is perfect).
Small, far-away road users in VisDrone are hard, so expect cars around 0.7–0.8 and pedestrians lower.

In [ ]:
best = YOLO(f"{PROJECT}/visdrone/weights/best.pt")
metrics = best.val(data="VisDrone.yaml", imgsz=960, split="val")
for j, c in enumerate(metrics.box.ap_class_index):     # only classes that appear in val
    print(f"{best.names[int(c)]:16s} mAP50 = {metrics.box.ap50[j]:.2f}")

## 6. Look at a few predictions

In [ ]:
import glob
from IPython.display import Image, display
samples = sorted(glob.glob("/content/datasets/VisDrone/images/val/*.jpg"))[:3]
for r in best.predict(samples, imgsz=960, conf=0.25, save=True, project="/content/preview", name="val", exist_ok=True):
    pass
for p in sorted(glob.glob("/content/preview/val/*.jpg"))[:3]:
    display(Image(p, width=900))

## 7. Download the model
Put `nearmiss-visdrone.pt` in the NearMiss folder (next to `app.py`).

In [ ]:
import shutil
from google.colab import files
shutil.copy(f"{PROJECT}/visdrone/weights/best.pt", "nearmiss-visdrone.pt")
files.download("nearmiss-visdrone.pt")